In [25]:
import pandas as pd
import numpy as np

def build_24_month_cohort(my_table_file, dxsum_file):
    print("Loading data...")
    df_my = pd.read_csv(my_table_file, low_memory=False)
    df_dx = pd.read_csv(dxsum_file, low_memory=False)
    
    # df_dx VISCODE columns to visit for consistency and also subject id if needed
    if 'VISCODE' in df_dx.columns:
        df_dx.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)

    # ---------------------------------------------------------
    # 1. Grab ALL Features from My_Table (Fixes Problem 1)
    # ---------------------------------------------------------
    # Sort to ensure 'bl' comes first, then take the first valid row per patient
    first_visits = df_my.sort_values(['subject_id', 'visit']).groupby('subject_id').first().reset_index()
    
    # Filter for baseline MCI patients
    mci_cohort = first_visits[first_visits['entry_research_group'].str.contains('MCI', na=False, case=False)].copy()
    mci_subjects = mci_cohort['subject_id'].unique()
    print(f"Identified {len(mci_subjects)} baseline MCI patients.")

    # Select the comprehensive feature list
    desired_features = [
        'subject_id', 'entry_age', 'PTGENDER', 'PTEDUCAT', 'GENOTYPE', 
        'TOTAL13', 'CDRSB', 'MMSCORE', 'FAQTOTAL', 'MOCA', 'NPISCORE'
    ]
    # Keep only what exists to prevent errors
    actual_features = [f for f in desired_features if f in mci_cohort.columns]
    mci_final_features = mci_cohort[actual_features].copy()

    # ---------------------------------------------------------
    # 2. 24-Month Window Logic (Fixes Problems 2 & 3)
    # ---------------------------------------------------------
    def get_month(v):
        v = str(v).lower().strip()
        if v in ['bl', 'sc']: return 0
        if v.startswith('m'):
            try: return int(v.replace('m', ''))
            except: return -1
        return -1
        
    df_dx['month'] = df_dx['visit'].apply(get_month)
    long_dx = df_dx[df_dx['subject_id'].isin(mci_subjects)]
    
    progressors = []
    stable = []
    
    PREDICTION_WINDOW = 24 # 24 Months (2 Years)
    
    for subj_id, group in long_dx.groupby('subject_id'):
        converted = False
        
        # Check ANY visit from Baseline up to Month 24
        window_24m = group[group['month'] <= PREDICTION_WINDOW]
        
        if 'DIAGNOSIS' in window_24m.columns and 3 in window_24m['DIAGNOSIS'].values: 
            converted = True
        if 'DXCHANGE' in window_24m.columns and any(x in [3, 5, 6] for x in window_24m['DXCHANGE'].values): 
            converted = True
            
        if converted:
            # They got AD *any time* within 24 months
            progressors.append(subj_id)
        else:
            # Did they stay in the study for AT LEAST 24 months to prove they were stable?
            if group['month'].max() >= PREDICTION_WINDOW:
                stable.append(subj_id)

    # ---------------------------------------------------------
    # 3. Apply Labels and Final Output
    # ---------------------------------------------------------
    final_df = mci_final_features[mci_final_features['subject_id'].isin(progressors + stable)].copy()
    final_df['label'] = final_df['subject_id'].apply(lambda x: 1 if x in progressors else 0)

    print("\n" + "="*50)
    print("24-MONTH MASTER DATASET READY")
    print("="*50)
    print(f"Total Valid Patients: {len(final_df)} (Progressors: {len(progressors)}, Stable: {len(stable)})")
    
    print("\nColumns in final dataset:")
    print(final_df.columns.tolist())
    
    return final_df

# --- RUN IT ---
my_table_path = '../data/adni/raw_data/All_Subjects_My_Table_27Mar2026.csv'
dxsum_name = '../data/adni/raw_data/All_Subjects_DXSUM_27Mar2026.csv'
final_df = build_24_month_cohort(my_table_path, dxsum_name)

Loading data...
Identified 1674 baseline MCI patients.

24-MONTH MASTER DATASET READY
Total Valid Patients: 543 (Progressors: 400, Stable: 143)

Columns in final dataset:
['subject_id', 'entry_age', 'PTGENDER', 'PTEDUCAT', 'GENOTYPE', 'TOTAL13', 'CDRSB', 'MMSCORE', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'label']


In [26]:
# prevalence on this cohort (only bl visit to 24 months visits)
print("\nPrevalence of AD Progression within 24 Months:")
print(final_df['label'].value_counts(normalize=True))


Prevalence of AD Progression within 24 Months:
label
1    0.736648
0    0.263352
Name: proportion, dtype: float64


In [27]:
import pandas as pd
import numpy as np

def build_rolling_window_cohort(date_suffix="27Mar2026", window_months=24):
    print(f"Building Rolling-Window Cohort ({window_months}-month horizon)...")
    
    df_my = pd.read_csv(f'../data/adni/raw_data/All_Subjects_My_Table_{date_suffix}.csv', low_memory=False)
    df_dx = pd.read_csv(f'../data/adni/raw_data/All_Subjects_DXSUM_{date_suffix}.csv', low_memory=False)
    df_demo = pd.read_csv(f'../data/adni/raw_data/All_Subjects_PTDEMOG_{date_suffix}.csv', low_memory=False)
    df_adas = pd.read_csv(f'../data/adni/raw_data/All_Subjects_ADAS_{date_suffix}.csv', low_memory=False)
    
    # rename visit and subject_id in dx if needed
    
    df_dx.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)
    # renam into demo and adas also
    df_demo.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)
    df_adas.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)

    # 1. Standardize the 'month' column across the board
    def get_month(v):
        v = str(v).lower().strip()
        if v in ['bl', 'sc']: return 0
        if v.startswith('m'):
            try: return int(v.replace('m', ''))
            except: return -1
        return -1

    df_my['month'] = df_my['visit'].apply(get_month)
    df_dx['month'] = df_dx['visit'].apply(get_month)
    df_adas['month'] = df_adas['visit'].apply(get_month)
    
    # 2. Get Static Demographics (Age, Gender, Education, APOE)
    # These don't change, so we just grab the baseline row for each patient
    print("Extracting static demographics...")
    bl_demo = df_demo.sort_values(['subject_id', 'visit']).groupby('subject_id').first().reset_index()
    demo_cols = [c for c in ['subject_id', 'PTGENDER', 'PTEDUCAT', 'entry_age', 'AGE', 'GENOTYPE', 'APOE4'] if c in bl_demo.columns]
    bl_demo = bl_demo[demo_cols]

    # 3. Create the Master Longitudinal Feature Table
    print("Fusing longitudinal clinical scores...")
    # Base it on My_Table since it has most scores
    long_features = df_my[['subject_id', 'visit', 'month', 'entry_research_group', 'MMSCORE', 'CDRSB', 'FAQTOTAL']].copy()
    
    # Merge ADAS-Cog longitudinally (matching on both subject_id AND month)
    if 'TOTAL13' in df_adas.columns:
        long_adas = df_adas[['subject_id', 'month', 'TOTAL13']].drop_duplicates(subset=['subject_id', 'month'])
        long_features = long_features.merge(long_adas, on=['subject_id', 'month'], how='left')
        
    # Broadcast static demographics to every longitudinal row
    long_features = long_features.merge(bl_demo, on='subject_id', how='left')

    # 4. The Rolling Window Label Generator
    print(f"Scanning the future {window_months} months for EVERY valid visit...")
    valid_rows = []
    
    # Group the diagnosis table by patient for fast future-lookup
    dx_grouped = dict(tuple(df_dx.groupby('subject_id')))
    
    # Iterate through every single visit in our feature table
    for idx, row in long_features.iterrows():
        subj = row['subject_id']
        current_month = row['month']
        
        # SKIP invalid months or missing diagnosis tracking
        if current_month < 0 or subj not in dx_grouped: continue
        
        patient_dx = dx_grouped[subj]
        
        # Check current diagnosis: We ONLY want to make predictions if they are CURRENTLY MCI
        # If they already converted to AD at this month, we don't use this row as a starting line!
        current_dx_row = patient_dx[patient_dx['month'] == current_month]
        is_currently_ad = False
        if not current_dx_row.empty:
            if 'DIAGNOSIS' in current_dx_row.columns and 3 in current_dx_row['DIAGNOSIS'].values: is_currently_ad = True
            if 'DXCHANGE' in current_dx_row.columns and any(x in [3, 4, 5, 6] for x in current_dx_row['DXCHANGE'].values): is_currently_ad = True
        
        if is_currently_ad:
            continue # Skip, they already have Alzheimer's
            
        # Define their personal "future" window for this specific visit
        target_month = current_month + window_months
        
        # Look at their diagnosis history strictly between (current_month) and (target_month)
        future_window = patient_dx[(patient_dx['month'] > current_month) & (patient_dx['month'] <= target_month)]
        
        converted = False
        if 'DIAGNOSIS' in future_window.columns and 3 in future_window['DIAGNOSIS'].values: converted = True
        if 'DXCHANGE' in future_window.columns and any(x in [3, 5, 6] for x in future_window['DXCHANGE'].values): converted = True
            
        if converted:
            row['label'] = 1
            valid_rows.append(row)
        else:
            # Did they actually stay in the study up to the target month?
            if patient_dx['month'].max() >= target_month:
                row['label'] = 0
                valid_rows.append(row)

    final_df = pd.DataFrame(valid_rows)
    
    print("\n" + "="*50)
    print("ROLLING-WINDOW DATASET READY")
    print("="*50)
    print(f"Total Unique Patients Used: {final_df['subject_id'].nunique()}")
    print(f"Total Training Rows Generated: {len(final_df)}")
    print(f"Class 1 (Progressors): {len(final_df[final_df['label'] == 1])}")
    print(f"Class 0 (Stable): {len(final_df[final_df['label'] == 0])}")
    print("-" * 50)
    print(f"Prior (Prevalence): {round(final_df['label'].mean(), 3)}")
    
    return final_df

# --- RUN IT ---
rolling_dataset = build_rolling_window_cohort(date_suffix="27Mar2026", window_months=24)

Building Rolling-Window Cohort (24-month horizon)...
Extracting static demographics...
Fusing longitudinal clinical scores...
Scanning the future 24 months for EVERY valid visit...



ROLLING-WINDOW DATASET READY
Total Unique Patients Used: 684
Total Training Rows Generated: 2979
Class 1 (Progressors): 997
Class 0 (Stable): 1982
--------------------------------------------------
Prior (Prevalence): 0.335


In [28]:
import pandas as pd

def prep_and_save_data(rolling_df, output_filename="adni_rolling_locf.csv"):
    df = rolling_df.copy()
    
    print("Applying patient-specific forward-fill (LOCF)...")
    # Sort chronologically so forward-fill works correctly
    df = df.sort_values(['subject_id', 'month'])
    
    # Safely impute longitudinally 
    clinical_scores = ['TOTAL13', 'MMSCORE', 'CDRSB', 'FAQTOTAL']
    df[clinical_scores] = df.groupby('subject_id')[clinical_scores].ffill()
    
    # Format Gender
    if df['PTGENDER'].dtype == 'O':
        df['PTGENDER'] = df['PTGENDER'].map({'Male': 0, 'Female': 1, 'M': 0, 'F': 1})
        df['PTGENDER'] = pd.to_numeric(df['PTGENDER'], errors='coerce').fillna(0)
        
    # SAVE IT! (It will still have a few NaNs for missing Day 1 data, which is perfect)
    df.to_csv(output_filename, index=False)
    print(f"Saved leak-free dataset to {output_filename}")
    
    return df

# --- RUN IT ---
final_df = prep_and_save_data(rolling_dataset, output_filename="../data/adni/adni_rolling_locf.csv")

Applying patient-specific forward-fill (LOCF)...
Saved leak-free dataset to ../data/adni/adni_rolling_locf.csv
